## Librerias

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

Importamos las librerias necesarias para cargar, analizar, visualizar y posteriormente dividir el conjunto de datos

## Semilla Aleatoria

Se define una semilla fija para garantizar que los procesos que involucren aleatoriedad sean reproducibles 

In [4]:
SEED=42
np.random.seed(SEED)


Al fijar la semilla en 42 garantizamos reproducibilidad, es decir, que otra persona pueda ejecutar el nootebook y obtener exactamente la misma division y de esta forma evitar resultados como Q100 -> train y en otra ejecucion Q100 -> validation 

## Carga de datos

In [5]:
df = pd.read_csv(
    "WikiQA_Tarea3.tsv",
    sep="\t"
)

Como el archivo no es realmente un csv separado por comnas si no un tsv, utilizamos sep="\t" para indicarle a pandas que cada tabulacion representa una nueva columna 

## Revision inicial de datos

Se revisan las dimensiones del conjunto, los nombres de las columnas, los tipos de datos y los primeros registros

In [6]:
print(df.shape)
# obtenemos el tamaño del dataset
print(df.columns.tolist())
# obtenemos los nombres de las columnas
print(df.dtypes)
# obtenemos los tipos de datos de cada columna
print(df.head())
# obtenemos las primeras 5 filas del dataset

(29196, 7)
['QuestionID', 'Question', 'DocumentID', 'DocumentTitle', 'SentenceID', 'Sentence', 'Label']
QuestionID         str
Question           str
DocumentID         str
DocumentTitle      str
SentenceID         str
Sentence           str
Label            int64
dtype: object
  QuestionID                     Question DocumentID  DocumentTitle  \
0       Q428  how much of US 75 is 4 lane       D423  U.S. Route 75   
1       Q428  how much of US 75 is 4 lane       D423  U.S. Route 75   
2       Q428  how much of US 75 is 4 lane       D423  U.S. Route 75   
3       Q428  how much of US 75 is 4 lane       D423  U.S. Route 75   
4       Q428  how much of US 75 is 4 lane       D423  U.S. Route 75   

  SentenceID                                           Sentence  Label  
0     D423-0      U.S. Route 75 is a north–south U.S. Highway .      0  
1     D423-1  The highway's northern terminus is in Kittson ...      0  
2     D423-2  Its southern terminus is at Interstate 30 and ...      0  
3 

El objetivo es comprobar que el archivo fue cargado correctamente y entender su estructura antes de realizar cualquier transformación.


## Cantidad de registros y preguntas 

Se calcula la cantidad total de registros y la cantidad de preguntas diferentes presentes en el dataset.

Debido a que una pregunta puede estar asociada a múltiples oraciones candidatas, ambas cantidades son diferentes.

In [7]:
print("Largo del data ser:", len(df))
# obtenemos el número de filas del dataset
print("Cantidad de preguntas únicas:",df["QuestionID"].nunique())
# obtenemos el número de preguntas únicas, es decir el número de filas que tienen un valor único en la columna "QuestionID"
print("Cantidad de documentos unicos:", df["DocumentID"].nunique())
# obtenemos el número de documentos únicos, es decir el número de filas que tienen un valor único en la columna "DocumentID"

Largo del data ser: 29196
Cantidad de preguntas únicas: 3043
Cantidad de documentos unicos: 2808


## Datos faltantes

Se verifica si existen valores ausentes en alguna de las columnas del dataset

In [8]:
datos_faltantes=df.isnull().sum()

print("La cantidad de datos faltantes es de:", datos_faltantes)

La cantidad de datos faltantes es de: QuestionID       0
Question         0
DocumentID       0
DocumentTitle    0
SentenceID       0
Sentence         0
Label            0
dtype: int64


Del resultado podes concluir que no existen datos faltantes

## Filas duplicadas

In [ ]:
duplicados_completos = df.duplicated().sum()

duplicados_id = df.duplicated(
    subset=["QuestionID", "SentenceID"] # trabajamos con el id de la pregunta
).sum()

print("Filas duplicadas:", duplicados_completos)
print("Duplicados QuestionID-SentenceID:", duplicados_id)   

Filas duplicadas: 0
Duplicados QuestionID-SentenceID: 0


No existen registros completamente repetidos ni identificadores de oración duplicados dentro de una misma pregunta. 

Por lo que no es necesario eliminar columnas.

## Oraciones repetidas

Además de revisar filas completamente duplicadas, se comprueba si una misma pregunta contiene más de una candidata con exactamente el mismo texto.

In [ ]:
duplicados_texto = df.duplicated(
    subset=["QuestionID", "Sentence"] # aqui no se trabaja con el id de la pregunta, sino con el texto de la pregunta
).sum()

print(
    "Candidatas textualmente repetidas:",
    duplicados_texto
)

Candidatas textualmente repetidas: 8


Existen 8 casos donde una pregunta posee candidatas cuyo texto es idéntico y apesar de esto no las eliminaremos, aunque se identificaron algunas oraciones textualmente repetidas dentro de una misma pregunta, estas no fueron eliminadas. 

Se debe a que corresponden a registros distintos del dataset original y poseen diferentes identificadores de oración. Además, su presencia no representa un impedimento significativo para las etapas posteriores de entrenamiento, recuperación y evaluación, ya que el sistema trabajará con las candidatas asociadas a cada `QuestionID` tal como fueron entregadas en el conjunto de datos.

## Cantidad de candidatas por pregunta